In [13]:
from pathlib import Path

import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---------- data path: ../data first, else search up the folder tree ----------
FILENAME = "plot1+2_debt-to-gdp_g20.csv"
PATH = Path("../data") / FILENAME
if not PATH.exists():
    PATH = next(
        p for parent in [Path.cwd(), *list(Path.cwd().parents)[:4]]
        for p in parent.rglob(FILENAME)
    )
print("Using:", PATH.resolve())

# ---------- settings ----------
COUNTRY = "Germany"
PROJECTIONS_FROM = 2026        # years from here are IMF projections (lighter line)
EVENT_COL = "#E6224B"
GREY = "#676A86"

# (year, label, which side of the red line the label sits on)
# events outside the plotted range are dropped automatically
EVENTS = [
    (1914, "WWI",             "left"),
    (1923, "Hyperinflation",  "left"),
    (1939, "WWII",            "left"),
    (1948, "Currency reform", "left"),
    (1990, "Reunification",   "left"),
    (1999, "Euro",     "left"),
    (2008, "GFC",             "right"),
    (2010, "Euro crisis",     "left"),
    (2020, "COVID",        "left"),
    (2025, "Debt brake",      "left"),
]

# ---------- data ----------
raw = pd.read_csv(PATH, na_values="no data")
df = raw[["Year", COUNTRY]].dropna().rename(columns={"Year": "year", COUNTRY: "debt"})
df["year"] = df["year"].astype(int)
df["date"] = pd.to_datetime(df["year"], format="%Y")
df["period"] = df["year"].ge(PROJECTIONS_FROM).map({True: "projection", False: "actual"})
df["series"] = COUNTRY          # constant field so the theme assigns the colour

# repeat the last actual year so the lighter projection joins the line
join = df[df["year"] == PROJECTIONS_FROM - 1].assign(period="projection")
plot_df = pd.concat([df, join], ignore_index=True)

first, last = int(df["year"].min()), int(df["year"].max())
events = pd.DataFrame(EVENTS, columns=["year", "label", "lab_align"])
events = events[events["year"].between(first - 1, last)].copy()
events["date"] = pd.to_datetime(events["year"], format="%Y")
events["ypx"] = 8               # label height in pixels from the top
print("Events shown:", events["label"].tolist())

x_start = min(first, int(events["year"].min())) - 1     # one year of padding on the left

# ---------- chart ----------
ev_rule = alt.Chart(events).mark_rule(
    color=EVENT_COL, strokeDash=[3, 3], strokeWidth=1.2).encode(x="date:T")

def ev_text(align, dx):
    return alt.Chart(events[events["lab_align"] == align]).mark_text(
        align=align, dx=dx, fontSize=10, fontWeight="bold", color=EVENT_COL).encode(
        x="date:T", y=alt.Y("ypx:Q", scale=None), text="label:N")

left_txt, right_txt = ev_text("left", 4), ev_text("right", -4)

line = alt.Chart(plot_df).mark_line(strokeWidth=1.8).encode(
    x=alt.X(
        "date:T", title=None,
        scale=alt.Scale(domain=[f"{x_start}-01-01", f"{last}-01-01"]),
        axis=alt.Axis(format="%Y", tickCount=alt.TimeIntervalStep(interval="year", step=5)),
    ),
    y=alt.Y(
        "debt:Q", scale=alt.Scale(zero=True),
        title="Public debt, share of GDP",
        axis=alt.Axis(labelExpr="datum.label + '%'"),
    ),
    color=alt.Color("series:N", legend=None),
    opacity=alt.Opacity(
        "period:N", legend=None,
        scale=alt.Scale(domain=["actual", "projection"], range=[1, 0.45]),
    ),
    order="date:T",
)

caption = alt.Title(
    text="Source: IMF World Economic Outlook",
    subtitle=[f"{COUNTRY}: general government gross debt, % of GDP, {first}–{last}. "
              "Lighter line shows IMF projections; red lines mark historic events."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color=GREY, subtitleColor=GREY, dy=12)

chart = (
    (ev_rule + left_txt + right_txt + line)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor=GREY, titleColor=GREY))

styles.save(chart, path=".", name="germany_debt_to_gdp", svg=True)
chart.save("germany_debt_to_gdp.png", scale_factor=2.0)
chart

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Peterson Blogs/Roger Schneider/data/plot1+2_debt-to-gdp_g20.csv
Events shown: ['Reunification', 'Euro', 'GFC', 'Euro crisis', 'COVID', 'Debt brake']


alt.LayerChart(...)